# 模型 4：MLP

**负责人：** 4 号（CAO Taihe）　**阶段：** 3 模型

句向量 → PyTorch MLP（从零训练）。实验设计来自 4 号的交付（`Member4_file/`）；`src/training.py` 和 `src/member4.py` 按他的说明（`TRAINING_GUIDE.md`）和测试（`test_member4.py`）补齐。

## 任务

标"（选做）"的任务在必做部分完成后再做。

- [x] 512 → 256 两层，BatchNorm、Dropout、AdamW、early stopping（按组分出的内层验证集选轮数）
- [x] FastText 输入要用对应折的版本：第 k 折用 `_fold{k}`，预测测试集用 `_full`
- [x] 每次实验先用 `new_run_id()` 生成 run_id，评分记录和预测文件都带上同一个 run_id
- [x] 对比三组参数（initial、regularized、weighted）和同折 TF-IDF + 逻辑回归参照
- [ ] 5 号的 SBERT 句向量到位后，换成 SBERT 输入再跑

## 产出

- `results/m4_mlp.csv`；OOF 概率和所选模型的测试集概率用 `save_preds` 保存（测试集不评分）
- `Member4_file/evidence/`（各折成绩、训练曲线、OOF 预测、概况 `summary.json`）
- `figures/m4_comparison.png`、`figures/m4_oof_confusion.png`、`figures/m4_learning_curves.png`

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

## 运行实验

约 1–2 分钟（CPU）：先按折训练 FastText，再对三组参数各做 5 折交叉验证。

In [ ]:
import os
os.environ.setdefault("MKL_THREADING_LAYER", "GNU")
import torch
torch.set_num_threads(2)

from src.member4 import run

summary = run()
print("selected:", summary["selected_parameters"]["name"], "| development OOF macro-F1:", round(summary["selected_cv"]["macro_f1"], 4))
assert summary["holdout_evaluated"] is False

In [ ]:
import pandas as pd
from IPython.display import Image, display

from src.member4 import EVIDENCE_DIR

display(pd.read_csv(EVIDENCE_DIR / "candidate_metrics.csv")[["run_id", "accuracy", "macro_precision", "macro_recall", "macro_f1", "train_seconds"]].round(4))
display(pd.read_csv(EVIDENCE_DIR / "baseline_reference.csv")[["model", "accuracy", "macro_f1"]].round(4))
for name in ["m4_comparison.png", "m4_oof_confusion.png", "m4_learning_curves.png"]:
    display(Image(filename=str(FIGURES_DIR / name)))